# Pipeline EIF — Índice de Interceptação de Borda (Transecto Linear por Grade Hexagonal)

**Fluxo:**
1. Configuração centralizada (edite apenas esta célula)
2. Instalação de dependências
3. Funções utilitárias
4. Extração de estatísticas por raster (com checkpoints)
5. Consolidação e exportação
6. Visualização (scatter + heatmap Área × Continuidade)

---
## 1. Configuração Centralizada
**Edite apenas esta célula.** O restante do notebook é genérico.

In [ ]:
# ============================================================
# CONFIGURAÇÃO — edite aqui e execute o notebook inteiro
# ============================================================

# Pasta raiz onde ficam os rasters .tif e o shapefile da grade
PASTA_DADOS = r"D:\Modelo_LAPIG\raster_TNC1_30"

# Shapefile da grade hexagonal (polígonos — serão convertidos para perímetros internamente)
SHAPEFILE_GRADE = r"D:\Modelo_LAPIG\raster_TNC1_30\grade_area_interesse.shp"

# Pasta de checkpoints (criada automaticamente)
PASTA_CHECKPOINTS = r"D:\Modelo_LAPIG\raster_TNC1_30\checkpoints"

# Arquivo de saída consolidado
CSV_SAIDA = r"D:\Modelo_LAPIG\raster_TNC1_30\eif_consolidado.csv"

# Valor nodata padrão (usado quando o raster não tem nodata no metadata)
NODATA_FALLBACK = 255

# Valor que representa vegetação/habitat nos rasters (pixel = 1)
VALOR_VEGETACAO = 1

# Mapeamento explícito: fragmento do nome do arquivo → rótulo do cenário
# (case-insensitive; primeiro match vence)
CENARIO_MAP = {
    "tnc1": "TNC1",
    "tnc2": "TNC2",
    "bau":  "BAU",
    "gov":  "GOV",
    "obs":  "OBS",   # observado / baseline
}

# Configurações do gráfico Área × Continuidade
# (usado na célula de visualização; pode ser alterado após a extração)
VIZ_CENARIO = "TNC1"
VIZ_ANO     = "2030"
VIZ_CSV_IN  = r"D:\Modelo_LAPIG\Gride\TNC1_2030_graf.csv"  # CSV com colunas prop_ e propC_
VIZ_OUT_DIR = r"D:\Modelo_LAPIG\Gride\outputs_area_cont"

---
## 2. Instalação de Dependências

In [ ]:
import importlib, subprocess, sys

PACOTES = ["rasterstats", "geopandas", "rasterio", "pandas", "numpy", "matplotlib"]

for pkg in PACOTES:
    if importlib.util.find_spec(pkg) is None:
        print(f"Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"✓ {pkg} já instalado")

---
## 3. Funções Utilitárias

In [ ]:
import os
import re
import glob
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

YEAR_RE = re.compile(r"(19|20)\d{2}")


def extrair_ano(nome_arquivo: str) -> str:
    """Extrai o primeiro ano no formato YYYY encontrado no nome do arquivo."""
    m = YEAR_RE.search(nome_arquivo)
    if not m:
        raise ValueError(f"Ano (YYYY) não encontrado em: {nome_arquivo}")
    return m.group(0)


def inferir_cenario(nome_arquivo: str, cenario_map: dict) -> str:
    """Infere o cenário a partir do nome do arquivo usando o mapeamento definido na config."""
    lower = nome_arquivo.lower()
    for chave, rotulo in cenario_map.items():
        if chave.lower() in lower:
            return rotulo
    return "DESCONHECIDO"


def garantir_id_unico(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Garante que o GeoDataFrame tenha coluna ID_UNICO."""
    if "ID_UNICO" not in gdf.columns:
        gdf = gdf.copy()
        gdf["ID_UNICO"] = gdf.index.astype(int)
    return gdf


def poligonos_para_perimetros(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Converte polígonos hexagonais em seus perímetros (linhas de borda).
    Isso é o que transforma cada hexágono em 6 transectos lineares sistemáticos.
    """
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.boundary
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    return gdf


def reprojetar_se_necessario(gdf: gpd.GeoDataFrame, raster_path: str) -> gpd.GeoDataFrame:
    """Reprojeta o GeoDataFrame para o CRS do raster, se forem diferentes."""
    with rasterio.open(raster_path) as src:
        crs_raster = src.crs
    if gdf.crs is None:
        raise ValueError("O shapefile não tem CRS definido.")
    if crs_raster is None:
        warnings.warn(f"Raster sem CRS: {raster_path}")
        return gdf
    if gdf.crs != crs_raster:
        return gdf.to_crs(crs_raster)
    return gdf


def obter_nodata(raster_path: str, fallback: int = 255) -> int:
    """Lê o nodata do metadata do raster; usa fallback se não definido."""
    with rasterio.open(raster_path) as src:
        nd = src.nodata
    if nd is None:
        warnings.warn(f"Raster sem nodata definido ({raster_path}). Usando fallback={fallback}.")
        return fallback
    return int(nd)


print("✓ Funções utilitárias carregadas.")

---
## 4. Extração do EIF por Raster

**Lógica do Índice de Interceptação de Borda (EIF):**
- O perímetro de cada hexágono é tratado como um transecto linear.
- `all_touched=True` garante que pixels tocados pela linha sejam contados.
- `categorical=True` retorna `{valor: contagem}`, evitando ambiguidade com `sum` em rasters binários.
- `px_tot` = total de pixels tocados pelo perímetro (excluindo nodata).
- `px_veg` = pixels com valor == VALOR_VEGETACAO tocados pelo perímetro.
- `prop_veg` = px_veg / px_tot → Índice de Interceptação (0 a 1).

In [ ]:
os.makedirs(PASTA_CHECKPOINTS, exist_ok=True)


def processar_raster(gdf_perimetros: gpd.GeoDataFrame, raster_path: str) -> pd.DataFrame:
    """
    Processa um único raster e retorna DataFrame com colunas:
      ID_UNICO, px_tot_{cenario}_{ano}, px_veg_{cenario}_{ano}, prop_veg_{cenario}_{ano}
    """
    base = os.path.basename(raster_path)
    ano = extrair_ano(base)
    cenario = inferir_cenario(base, CENARIO_MAP)
    tag = f"{cenario}_{ano}"

    checkpoint = os.path.join(PASTA_CHECKPOINTS, f"eif_{tag}.csv")
    if os.path.exists(checkpoint):
        print(f"  [checkpoint] {tag} — carregando {checkpoint}")
        return pd.read_csv(checkpoint)

    print(f"  [processando] {tag} | {base}")

    # Reprojetar para o CRS do raster
    gdf_proj = reprojetar_se_necessario(gdf_perimetros, raster_path)
    nodata = obter_nodata(raster_path, NODATA_FALLBACK)

    # Extração categórica: {valor_pixel: nº de pixels tocados}
    stats = zonal_stats(
        gdf_proj,
        raster_path,
        categorical=True,
        nodata=nodata,
        all_touched=True,   # essencial para geometrias lineares
    )

    # Montar DataFrame de resultados
    ids     = gdf_perimetros["ID_UNICO"].values
    px_veg  = np.array([s.get(VALOR_VEGETACAO, 0) or 0 for s in stats], dtype=float)
    px_tot  = np.array([sum(v for k, v in s.items() if k != nodata) for s in stats], dtype=float)

    with np.errstate(invalid="ignore", divide="ignore"):
        prop = np.where(px_tot > 0, px_veg / px_tot, np.nan)

    df_out = pd.DataFrame({
        "ID_UNICO":          ids,
        f"px_tot_{tag}":     px_tot.astype(int),
        f"px_veg_{tag}":     px_veg.astype(int),
        f"prop_veg_{tag}":   np.round(prop, 4),
    })

    df_out.to_csv(checkpoint, index=False)
    print(f"  [salvo] {checkpoint}")
    return df_out


# --- EXECUÇÃO PRINCIPAL ---

print("Carregando grade hexagonal...")
gdf_grade = gpd.read_file(SHAPEFILE_GRADE)
gdf_grade = garantir_id_unico(gdf_grade)
print(f"  {len(gdf_grade):,} hexágonos carregados | CRS: {gdf_grade.crs}")

print("\nConvertendo polígonos para perímetros (transectos lineares)...")
gdf_perimetros = poligonos_para_perimetros(gdf_grade)
print(f"  {len(gdf_perimetros):,} feições de perímetro geradas")

# Listar rasters
arquivos_raster = sorted(glob.glob(os.path.join(PASTA_DADOS, "*.tif")))
print(f"\n{len(arquivos_raster)} raster(s) encontrado(s) em {PASTA_DADOS}")

if not arquivos_raster:
    raise FileNotFoundError(f"Nenhum .tif encontrado em {PASTA_DADOS}")

# Processar cada raster e consolidar
df_final = gdf_grade[["ID_UNICO"]].copy()

print("\nIniciando extração...")
for raster_path in arquivos_raster:
    df_ano = processar_raster(gdf_perimetros, raster_path)
    df_final = df_final.merge(df_ano, on="ID_UNICO", how="left")

print(f"\n✓ Processamento concluído. Shape final: {df_final.shape}")

---
## 5. Exportação Consolidada

In [ ]:
df_final.to_csv(CSV_SAIDA, index=False, encoding="utf-8-sig")
print(f"✓ CSV consolidado salvo em: {CSV_SAIDA}")
print(f"  Linhas: {len(df_final):,} | Colunas: {len(df_final.columns)}")
df_final.head()

---
## 6. Visualização — Área × Continuidade

Gera dois outputs para o cenário e ano definidos em `VIZ_CENARIO` / `VIZ_ANO`:
- **Scatter plot** com grade de classes de 20%
- **Heatmap 5×5** com percentual de hexágonos em cada combinação de classe

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

os.makedirs(VIZ_OUT_DIR, exist_ok=True)

COL_AREA = f"prop_{VIZ_CENARIO}_{VIZ_ANO}"
COL_CONT = f"propC_{VIZ_CENARIO}_{VIZ_ANO}"
BIN_EDGES = np.array([0, 20, 40, 60, 80, 100], dtype=float)
BIN_LABELS = [f"{int(BIN_EDGES[i])}-{int(BIN_EDGES[i+1])}%" for i in range(5)]

# --- Carregar CSV de visualização ---
df_viz = pd.read_csv(VIZ_CSV_IN, sep=";", decimal=",")

for col in [COL_AREA, COL_CONT]:
    if col not in df_viz.columns:
        raise ValueError(f"Coluna '{col}' não encontrada.\nColunas disponíveis: {list(df_viz.columns)}")

x = df_viz[COL_CONT].astype(float).to_numpy()
y = df_viz[COL_AREA].astype(float).to_numpy()

# Normalizar para % se estiver em proporção (0-1)
if np.nanmax([np.nanmax(x), np.nanmax(y)]) <= 1.01:
    x, y = x * 100.0, y * 100.0

# Remover inválidos e clipar bordas
mask = np.isfinite(x) & np.isfinite(y)
x, y = np.clip(x[mask], 0, 100 - 1e-9), np.clip(y[mask], 0, 100 - 1e-9)

# Binning
xb = np.clip(np.digitize(x, BIN_EDGES, right=False) - 1, 0, 4)
yb = np.clip(np.digitize(y, BIN_EDGES, right=False) - 1, 0, 4)

counts = np.zeros((5, 5), dtype=int)
for i, j in zip(yb, xb):
    counts[i, j] += 1
perc = (counts / counts.sum()) * 100.0

titulo_base = f"{VIZ_CENARIO} — {VIZ_ANO}"

# --- Plot 1: Scatter com grade ---
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(x, y, s=4, alpha=0.35, color="steelblue", rasterized=True)
for b in BIN_EDGES:
    ax.axvline(b, color="gray", linewidth=0.8, linestyle="--")
    ax.axhline(b, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_xlabel("Continuidade (%)", fontsize=12)
ax.set_ylabel("Área (%)", fontsize=12)
ax.set_title(f"Área × Continuidade | {titulo_base}", fontsize=13)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%g%%"))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g%%"))
scatter_path = os.path.join(VIZ_OUT_DIR, f"scatter_{VIZ_CENARIO}_{VIZ_ANO}.png")
plt.savefig(scatter_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Scatter salvo: {scatter_path}")

# --- Plot 2: Heatmap 5×5 ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(perc, origin="lower", aspect="auto", cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="% de hexágonos")
ax.set_xticks(range(5))
ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right")
ax.set_yticks(range(5))
ax.set_yticklabels(BIN_LABELS)
ax.set_xlabel("Continuidade (%)", fontsize=12)
ax.set_ylabel("Área (%)", fontsize=12)
ax.set_title(f"Distribuição por classe (25 células 20×20%) | {titulo_base}", fontsize=12)
for i in range(5):
    for j in range(5):
        cor = "white" if perc[i, j] > perc.max() * 0.6 else "black"
        ax.text(j, i, f"{perc[i, j]:.1f}%", ha="center", va="center", fontsize=9, color=cor)
heatmap_path = os.path.join(VIZ_OUT_DIR, f"heatmap_{VIZ_CENARIO}_{VIZ_ANO}.png")
plt.savefig(heatmap_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Heatmap salvo: {heatmap_path}")

# --- CSVs auxiliares ---
pd.DataFrame(counts, index=BIN_LABELS, columns=BIN_LABELS).to_csv(
    os.path.join(VIZ_OUT_DIR, f"counts_{VIZ_CENARIO}_{VIZ_ANO}.csv"), encoding="utf-8-sig")
pd.DataFrame(np.round(perc, 2), index=BIN_LABELS, columns=BIN_LABELS).to_csv(
    os.path.join(VIZ_OUT_DIR, f"percent_{VIZ_CENARIO}_{VIZ_ANO}.csv"), encoding="utf-8-sig")

# Formato long
records = [
    {"cenario": VIZ_CENARIO, "ano": VIZ_ANO,
     "classe_area": BIN_LABELS[i], "classe_continuidade": BIN_LABELS[j],
     "count": int(counts[i, j]), "percent": round(float(perc[i, j]), 4)}
    for i in range(5) for j in range(5)
]
pd.DataFrame(records).to_csv(
    os.path.join(VIZ_OUT_DIR, f"long_{VIZ_CENARIO}_{VIZ_ANO}.csv"), index=False, encoding="utf-8-sig")

print("\n✓ CSVs (counts, percent, long) salvos em:", VIZ_OUT_DIR)